# EDA LOAN

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import style

import seaborn as sns

%matplotlib inline

In [ ]:
import warnings
warnings.filterwarnings("ignore")

style.use('ggplot') or plt.style.use('ggplot')

In [ ]:
df_loan = pd.read_csv('./data_loan.csv')

df_loan.sample(n=6)

In [ ]:
df_loan.info()

#### Eliminar campos ID por no necesarios

In [ ]:
df_loan.drop(columns=['Loan ID', 'Customer ID'], inplace=True)

#### Cambio nombre columnas (sin espacios)

In [ ]:
df_loan.columns = [columna.replace(' ','_') for columna in df_loan.columns.to_list()]

df_loan.columns

In [ ]:
df_loan.describe()

In [ ]:
df_loan.describe(include='object')

In [ ]:
df_loan.duplicated().sum()

#### Eliminamos duplicados

In [ ]:
df_loan.drop_duplicates(inplace=True)

In [ ]:
var_num = df_loan.select_dtypes(exclude='object').columns.to_list()

var_cat = df_loan.select_dtypes(include='object').columns.to_list()


## ANALISIS UNIDIMENSIONAL

In [ ]:
# valores colormap
TABLEAU_CMP = ('tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', \
               'tab:gray','tab:olive', 'tab:cyan')

In [ ]:
# creamos la grafica
fig, axes = plt.subplots(len(var_num), 2, \
                         figsize=(20, 5 * len(var_num)), \
                         gridspec_kw={'hspace': 0.4, 'wspace': 0.1})
ax = axes.ravel()

# graficas distribucion y boxplot de cada atributo
for idx, atributo in enumerate(var_num):

    # distribucion (histograma)
    sns.distplot(df_loan[atributo], bins=30, ax=ax[2 * idx], \
                 color=TABLEAU_CMP[idx % len(TABLEAU_CMP)], \
                 hist_kws={'alpha': 0.15})

    # titulo, etiquetas histograma
    ax[2 * idx].set_title(f'HISTOGRAMA {atributo}')
    ax[2 * idx].set_xlabel(f'Valores {atributo}')
    ax[2 * idx].set_ylabel("Frequencia")

    # boxplot
    sns.boxplot(x=atributo, data=df_loan, ax=ax[2 * idx + 1], color=TABLEAU_CMP[idx % len(TABLEAU_CMP)])

    # titulo, etiquetas boxplot
    ax[2 * idx + 1].set_title(f'BOXPLOT {atributo}')
    ax[2 * idx + 1].set_xlabel(f'Valores {atributo}')


# guardamos grafica:
fig.savefig('numericas')

##### Columnas "Problemas credito" y "Bancarrota", sopesar si ampliar rango intercuartilico o incluso eliminar

##### Columna "cantidad credito", inspeccionar fila del único atípico. Mima cuestión para "Ingresos" y "Nº de cuentas, con el añadido de presencia de otros outliers + separados

## UNIDIMENSIONALES CATEGORICOS

In [ ]:
len(var_cat)

In [ ]:
var_cat

In [ ]:
colores = sns.color_palette("husl", len(var_cat))


In [ ]:
# creacion graficas
fig, axes = plt.subplots(len(var_cat), 1, \
                         figsize=(10, 5*len(var_cat)),\
                         gridspec_kw={'hspace': 0.4, 'wspace': 0.4})

ax = axes.ravel()

# dibujamos las graficas
for idx,variable in enumerate(var_cat):

    # utilizar el método de dibujo que nos interese en cada momento
    # sustituir nombre dataframe y parámetros según método de dibujo

    sns.countplot(df_loan[variable], ax=ax[idx],\
                  palette=colores)

    ax[idx].set_title(f'HISTOGRAMA {variable}')
    ax[idx].set_xlabel(f'Valores atributo {variable}')
    ax[idx].set_ylabel("Frequencia")



En esta última llevar a otra categoria o eliminar "haveMortgage"

Ver "Años trabajo" en mas detalle:

In [ ]:
columna_anios_ordenada = df_loan.Years_in_current_job.sort_values()

columna_anios_ordenada

In [ ]:
sns.countplot(y=columna_anios_ordenada);

In [ ]:
df_loan.Years_in_current_job.unique()

#### Vamos a reorganizar la categórica "Años de antiguedad":

In [ ]:
df_loan.Years_in_current_job.replace({'6 years': '6+ years', \
                                      '7 years': '6+ years', \
                                      '8 years': '6+ years', \
                                      '9 years': '6+ years'}).value_counts()


La nueva categoria es demasiado alto el conteo, con lo cual una agrupacion de pequeños lo convierto en grande: **ERROR, falseo realidad.**

A partir de 6 años, esta vez voy a agrupar de 2 en 2

In [ ]:
df_loan.Years_in_current_job.replace({'6 years': '6_7 years', \
                                      '7 years': '6_7 years'}, \
                                    inplace=True)


In [ ]:
df_loan.Years_in_current_job.replace({'8 years': '8_9 years', \
                                      '9 years': '8_9 years'}, \
                                    inplace=True)

df_loan.Years_in_current_job.value_counts()


### Categoria casi no apreciable "HaveMortgage" en categórica "Home_Ownership"

In [ ]:
df_loan.Home_Ownership.value_counts()

In [ ]:
df_home_minoritario = df_loan[df_loan.Home_Ownership=='HaveMortgage']

len(df_home_minoritario)

**Comparamos distribuciones:**

In [ ]:
df_home_minoritario.describe()

In [ ]:
df_loan.describe()

Alguna diferencia mínima, acaso estariamos eliminando de outliers. Veamos el target:

In [ ]:
df_home_minoritario.Loan_Status.value_counts()

De nuevo mayoritaria "Fully Paid" y separada.
#### Vamos entonces a eliminar las filas de esa categoria minima:

In [ ]:
df_loan = df_loan.drop(index=df_loan[df_loan.Home_Ownership=='HaveMortgage'].index)

sns.countplot(df_loan.Home_Ownership)

In [ ]:
len(df_loan)

## ANÁLISIS BIDIMENSIONAL

### Categórico-categorico.

In [ ]:
var_cat

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))

sns.countplot(x='Years_in_current_job', hue='Loan_Status', \
              data=df_loan);

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))

sns.countplot(x='Home_Ownership', hue='Loan_Status', \
              data=df_loan);

Parece que **"antiguedad trabajo" no tendria influencia en el target, sin embargo "Home Ownership" si.**

### Numérico-numérico.

In [ ]:
sns.pairplot(data=df_loan, hue='Loan_Status', diag_kind='kde');

In [ ]:
# recuerden 'numeric_only=True'
mat_corr = df_loan.corr()

mat_corr


In [ ]:
sns.heatmap(mat_corr, annot=True, cbar=True, \
            cmap=sns.color_palette("coolwarm", as_cmap=True))
plt.title('Correlación entre atributos');

**Poco que decir, acaso la  esperada entre Problemas Crediticios y Bancarrotas. Podríamos eliminar una de ellas**

**OJO**: Información de la diagonal del pairplot

### Numérico-categorico.

In [ ]:
var_num

In [ ]:
var_cat

In [ ]:
fig, axes = plt.subplots(len(var_num), 1, figsize=(10, 3*len(var_num)), gridspec_kw={'hspace': 0.4, 'wspace': 0.4})

ax = axes.ravel()

# dibujamos las graficas
for idx,variable in enumerate(var_num):

    sns.boxplot(data=df_loan, x=variable, y='Loan_Status')

    ax[idx].set_title(f'LOAN STATUS  vs {variable}')
    ax[idx].set_xlabel(f'Valores atributo {variable}')


#### Habremos de esperar a corregir los outliers para el bidimensional numérico-categorico